In [7]:
!pip install transformers datasets huggingface_hub sentencepiece emoji -q

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Device: cuda
GPU: Tesla T4


In [8]:
# Cell 2 — Load all datasets and combine

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        print(os.path.join(dirname, f))

/kaggle/input/datasets/sourabhsaxena/hatespeech01/hi_Hasoc2021_train.csv
/kaggle/input/datasets/sourabhsaxena/hatespeech01/val.csv
/kaggle/input/datasets/sourabhsaxena/hatespeech01/hindi_dataset.tsv
/kaggle/input/datasets/sourabhsaxena/hatespeech01/hasoc2019_hi_test_gold_2919.tsv
/kaggle/input/datasets/sourabhsaxena/hatespeech01/train.csv
/kaggle/input/datasets/sourabhsaxena/hatespeech01/test.csv
/kaggle/input/datasets/sourabhsaxena/hatespeech01/hi_Hasoc2021_test_task1.csv


In [9]:
# ── Load original dataset ──────────────────────────────
BASE = '/kaggle/input/datasets/sourabhsaxena/hatespeech01'
orig_train = pd.read_csv(f'{BASE}/train.csv')
orig_val   = pd.read_csv(f'{BASE}/val.csv')
orig_test  = pd.read_csv(f'{BASE}/test.csv')

orig_df = pd.concat([orig_train, orig_val], ignore_index=True)
orig_df = orig_df[['text', 'label']].copy()
orig_df['source'] = 'hinglish_original'
print(f"Original dataset: {len(orig_df)} samples")

# ── Load HASOC 2019 Train ──────────────────────────────
HASOC_PATH = '/kaggle/input/datasets/sourabhsaxena/hatespeech01'  # update after Cell 2

hasoc19_train = pd.read_csv(
    f'{HASOC_PATH}/hindi_dataset.tsv', sep='\t'
)
hasoc19_train = pd.DataFrame({
    'text'  : hasoc19_train['text'].astype(str),
    'label' : hasoc19_train['task_1'].map({'HOF': 1, 'NOT': 0}),
    'source': 'hasoc2019_train'
})
hasoc19_train = hasoc19_train.dropna()
print(f"HASOC 2019 Train: {len(hasoc19_train)} samples")

# ── Load HASOC 2019 Test ───────────────────────────────
hasoc19_test = pd.read_csv(
    f'{HASOC_PATH}/hasoc2019_hi_test_gold_2919.tsv', sep='\t'
)
hasoc19_test = pd.DataFrame({
    'text'  : hasoc19_test['text'].astype(str),
    'label' : hasoc19_test['task_1'].map({'HOF': 1, 'NOT': 0}),
    'source': 'hasoc2019_test'
})
hasoc19_test = hasoc19_test.dropna()
print(f"HASOC 2019 Test : {len(hasoc19_test)} samples")

# ── Load HASOC 2021 Train ──────────────────────────────
hasoc21_train = pd.read_csv(f'{HASOC_PATH}/hi_Hasoc2021_train.csv')
hasoc21_train = pd.DataFrame({
    'text'  : hasoc21_train['text'].astype(str),
    'label' : hasoc21_train['task_1'].map({'HOF': 1, 'NOT': 0}),
    'source': 'hasoc2021_train'
})
hasoc21_train = hasoc21_train.dropna()
print(f"HASOC 2021 Train: {len(hasoc21_train)} samples")

# ── Combine all ────────────────────────────────────────
all_df = pd.concat([
    orig_df,
    hasoc19_train,
    hasoc19_test,
    hasoc21_train
], ignore_index=True)

# Clean
all_df['text']  = all_df['text'].astype(str).str.strip()
all_df['label'] = all_df['label'].astype(int)
all_df = all_df.dropna()
all_df = all_df.drop_duplicates(subset=['text'])
all_df = all_df[all_df['text'].str.len() > 3]

print(f"\n{'='*50}")
print(f"COMBINED DATASET SUMMARY")
print(f"{'='*50}")
print(f"Total samples : {len(all_df)}")
print(f"\nBy source:")
print(all_df['source'].value_counts())
print(f"\nLabel distribution:")
print(all_df['label'].value_counts())
ratio = all_df['label'].value_counts().max() / all_df['label'].value_counts().min()
print(f"\nImbalance ratio: {ratio:.2f}x")

Original dataset: 23626 samples
HASOC 2019 Train: 4665 samples
HASOC 2019 Test : 1318 samples
HASOC 2021 Train: 4594 samples

COMBINED DATASET SUMMARY
Total samples : 33944

By source:
source
hinglish_original    23594
hasoc2019_train       4651
hasoc2021_train       4383
hasoc2019_test        1316
Name: count, dtype: int64

Label distribution:
label
0    18523
1    15421
Name: count, dtype: int64

Imbalance ratio: 1.20x


In [10]:
# Cell 3 — Train/Val split
# Keep original test set for fair comparison!

SEED = 42

# Use original test set for evaluation
test_df = orig_test.copy()
test_df['text']  = test_df['text'].astype(str)
test_df['label'] = test_df['label'].astype(int)

# Split combined data into train/val
train_df, val_df = train_test_split(
    all_df,
    test_size=0.1,
    random_state=SEED,
    stratify=all_df['label']
)

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)} (original, unchanged)")
print(f"\nTrain label dist:\n{train_df['label'].value_counts()}")

Train : 30549
Val   : 3395
Test  : 5907 (original, unchanged)

Train label dist:
label
0    16670
1    13879
Name: count, dtype: int64


In [11]:
# Cell 4 — Round 3 Config
CONFIG = {
    'model_name'    : 'google/muril-base-cased',
    'num_labels'    : 2,
    'max_length'    : 128,
    'batch_size'    : 32,
    'epochs'        : 5,
    'learning_rate' : 2e-5,
    'warmup_ratio'  : 0.1,
    'weight_decay'  : 0.01,
    'patience'      : 3,
    'label2id'      : {'NOT': 0, 'HOF': 1},
    'id2label'      : {0: 'NOT', 1: 'HOF'},
    'output_dir'    : '/kaggle/working/muril_r3',
    'hf_repo'       : 'sourabh5500/hate-speech-muril',
    'seed'          : 42,
}

import random
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
torch.cuda.manual_seed_all(CONFIG['seed'])

print("✅ Round 3 Config ready")
print(f"Key changes vs Round 1:")
print(f"  Training data: 23,626 → {len(train_df)} samples")
print(f"  Added HASOC 2019 + 2021 Hindi data")

✅ Round 3 Config ready
Key changes vs Round 1:
  Training data: 23,626 → 30549 samples
  Added HASOC 2019 + 2021 Hindi data


In [12]:
# Cell 5
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

class HateSpeechDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length= max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(),
            'attention_mask' : enc['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = HateSpeechDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer, CONFIG['max_length'])
val_dataset   = HateSpeechDataset(val_df['text'].tolist(),   val_df['label'].tolist(),   tokenizer, CONFIG['max_length'])
test_dataset  = HateSpeechDataset(test_df['text'].tolist(),  test_df['label'].tolist(),  tokenizer, CONFIG['max_length'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")
print("✅ DataLoaders ready")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Train batches: 955
Val batches  : 54
Test batches : 93
✅ DataLoaders ready


In [13]:
# Cell 6
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    id2label=CONFIG['id2label'],
    label2id=CONFIG['label2id'],
)
model = model.to(device)

no_decay = ['bias', 'LayerNorm.weight']
optimizer_params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': CONFIG['weight_decay']},
    {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer    = AdamW(optimizer_params, lr=CONFIG['learning_rate'])
total_steps  = len(train_loader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")
print("✅ Model + Optimizer ready")

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Total steps  : 4775
Warmup steps : 477
✅ Model + Optimizer ready


In [14]:
# Cell 7
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        total_loss += loss.item()
        if (batch_idx+1) % 100 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}")
    return total_loss/len(loader), f1_score(all_labels, all_preds, average='macro')

def evaluate(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            preds   = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            total_loss += outputs.loss.item()
    return total_loss/len(loader), f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels

print("✅ Functions ready")

✅ Functions ready


In [15]:
# Cell 8 — Training
import os
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("="*60)
print("MURIL ROUND 3 — MORE DATA TRAINING")
print("="*60)
print(f"Train samples : {len(train_df)} (was 23,626)")
print(f"Added         : HASOC 2019 + 2021 Hindi")
print(f"Epochs        : {CONFIG['epochs']}")
print("="*60)

best_val_f1, best_epoch, patience_count = 0.0, 0, 0
history = []

for epoch in range(1, CONFIG['epochs']+1):
    print(f"\n── Epoch {epoch}/{CONFIG['epochs']} ──────────────")
    train_loss, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_f1, _, _ = evaluate(model, val_loader, device)

    history.append({'epoch':epoch, 'train_loss':train_loss, 'train_f1':train_f1, 'val_loss':val_loss, 'val_f1':val_f1})
    print(f"\n  Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"  Val Loss  : {val_loss:.4f} | Val F1  : {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1, best_epoch, patience_count = val_f1, epoch, 0
        model.save_pretrained(CONFIG['output_dir'])
        tokenizer.save_pretrained(CONFIG['output_dir'])
        print(f"  ✅ New best saved! (Val F1: {val_f1:.4f})")
    else:
        patience_count += 1
        print(f"  ⏭ No improvement ({patience_count}/{CONFIG['patience']}) | Best: {best_val_f1:.4f}")
        if patience_count >= CONFIG['patience']:
            print(f"  🛑 Early stopping at epoch {epoch}")
            break

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE — Best Val F1: {best_val_f1:.4f} (Epoch {best_epoch})")
print(f"{'='*60}")

MURIL ROUND 3 — MORE DATA TRAINING
Train samples : 30549 (was 23,626)
Added         : HASOC 2019 + 2021 Hindi
Epochs        : 5

── Epoch 1/5 ──────────────
  Batch 100/955 | Loss: 0.6919
  Batch 200/955 | Loss: 0.6639
  Batch 300/955 | Loss: 0.6346
  Batch 400/955 | Loss: 0.7008
  Batch 500/955 | Loss: 0.6069
  Batch 600/955 | Loss: 0.5532
  Batch 700/955 | Loss: 0.5204
  Batch 800/955 | Loss: 0.5194
  Batch 900/955 | Loss: 0.6660

  Train Loss: 0.6137 | Train F1: 0.6485
  Val Loss  : 0.5227 | Val F1  : 0.7408


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!
  ✅ New best saved! (Val F1: 0.7408)

── Epoch 2/5 ──────────────
  Batch 100/955 | Loss: 0.3898
  Batch 200/955 | Loss: 0.6229
  Batch 300/955 | Loss: 0.4421
  Batch 400/955 | Loss: 0.3264
  Batch 500/955 | Loss: 0.4353
  Batch 600/955 | Loss: 0.5267
  Batch 700/955 | Loss: 0.4231
  Batch 800/955 | Loss: 0.6688
  Batch 900/955 | Loss: 0.4286

  Train Loss: 0.4892 | Train F1: 0.7644
  Val Loss  : 0.4754 | Val F1  : 0.7577


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ New best saved! (Val F1: 0.7577)

── Epoch 3/5 ──────────────
The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!
  Batch 100/955 | Loss: 0.4762
  Batch 200/955 | Loss: 0.4348
  Batch 300/955 | Loss: 0.3739
  Batch 400/955 | Loss: 0.5751
  Batch 500/955 | Loss: 0.3496
  Batch 600/955 | Loss: 0.4139
  Batch 700/955 | Loss: 0.2852
  Batch 800/955 | Loss: 0.4301
  Batch 900/955 | Loss: 0.3410

  Train Loss: 0.4003 | Train F1: 0.8154
  Val Loss  : 0.5009 | Val F1  : 0.7561
  ⏭ No improvement (1/3) | Best: 0.7577

── Epoch 4/5 ──────────────
  Batch 100/955 | Loss: 0.3536
  Batch 200/955 | Loss: 0.3114
  Batch 300/955 | Loss: 0.3527
  Batch 400/955 | Loss: 0.4630
  Batch 500/955 | Loss: 0.3042
  Batch 600/955 | Loss: 0.2620
  Batch 700/955 | Loss: 0.2248
  Batch 800

In [16]:
# Cell 9 — Test Evaluation Only (no HF push)

from transformers import AutoModelForSequenceClassification
from sklearn.metrics import f1_score
import numpy as np

ROUND1_F1 = 0.7352  # Current best

# Load best saved model
best_model = AutoModelForSequenceClassification.from_pretrained(CONFIG['output_dir'])
best_model = best_model.to(device)

# Evaluate
_, test_f1, test_preds, test_labels = evaluate(best_model, test_loader, device)

print("="*60)
print("ROUND 3 — FINAL TEST RESULTS")
print("="*60)
print(classification_report(
    test_labels, test_preds,
    target_names=['NOT', 'HOF'],
    digits=4
))

fn      = ((np.array(test_preds)==0) & (np.array(test_labels)==1)).sum()
fp      = ((np.array(test_preds)==1) & (np.array(test_labels)==0)).sum()
fn_rate = fn / (np.array(test_labels)==1).sum()
fp_rate = fp / (np.array(test_labels)==0).sum()
hof_recall = f1_score(test_labels, test_preds, pos_label=1, average='binary')

print(f"Test Macro-F1 : {test_f1:.4f}")
print(f"Round 1 F1    : {ROUND1_F1}")
print(f"Difference    : {test_f1-ROUND1_F1:+.4f}")

print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)
print(f"{'Metric':<20} | {'Round 1':>10} | {'Round 3':>10} | {'Change':>10}")
print("-"*56)
print(f"{'Macro F1':<20} | {0.7352:>10.4f} | {test_f1:>10.4f} | {test_f1-0.7352:>+10.4f}")
print(f"{'HOF Recall':<20} | {0.6668:>10.4f} | {hof_recall:>10.4f} | {hof_recall-0.6668:>+10.4f}")
print(f"{'FN Rate':<20} | {0.3376:>10.2%} | {fn_rate:>10.2%} | {fn_rate-0.3376:>+10.2%}")
print(f"{'FP Rate':<20} | {0.2058:>10.2%} | {fp_rate:>10.2%} | {fp_rate-0.2058:>+10.2%}")
print(f"{'FN Count':<20} | {'926':>10} | {fn:>10} | {fn-926:>+10}")

print("\n" + "="*60)
if test_f1 > ROUND1_F1:
    print(f"✅ Round 3 BETTER than Round 1!")
    print(f"   Recommendation: PUSH TO HF ✅")
else:
    print(f"❌ Round 3 NOT better than Round 1")
    print(f"   Recommendation: KEEP Round 1 on HF")
print("="*60)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ROUND 3 — FINAL TEST RESULTS
              precision    recall  f1-score   support

         NOT     0.7397    0.7724    0.7557      3164
         HOF     0.7234    0.6865    0.7045      2743

    accuracy                         0.7325      5907
   macro avg     0.7316    0.7295    0.7301      5907
weighted avg     0.7321    0.7325    0.7319      5907

Test Macro-F1 : 0.7301
Round 1 F1    : 0.7352
Difference    : -0.0051

COMPARISON TABLE
Metric               |    Round 1 |    Round 3 |     Change
--------------------------------------------------------
Macro F1             |     0.7352 |     0.7301 |    -0.0051
HOF Recall           |     0.6668 |     0.7045 |    +0.0377
FN Rate              |     33.76% |     31.35% |     -2.41%
FP Rate              |     20.58% |     22.76% |     +2.18%
FN Count             |        926 |        860 |        -66

❌ Round 3 NOT better than Round 1
   Recommendation: KEEP Round 1 on HF
